In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import Dataset, DataLoader
from transformers import BartForConditionalGeneration, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import evaluate
import wandb
from torch.amp import autocast, GradScaler

class Config:
    def __init__(self):
        # Resolve paths based on execution directory
        base = '.'
        if not os.path.exists('data') and os.path.exists('../data'):
            base = '..'
            
        self.excel_path = os.path.join(base, 'data/VideoQuestions.xlsx')
        self.sheet_name = 'Dropped Instances With NER Scor'
        self.clip_path = os.path.join(base, 'embeddings/QCLIP')
        self.eff_path = os.path.join(base, 'embeddings/QEfficient')
        
        # Architecture Settings
        self.bart_model = 'facebook/bart-base'
        self.d_model = 768
        self.num_heads = 8
        
        # Quantum Multi-Stream Settings
        self.n_streams = 8
        self.qubits_per_path = 4
        self.gates_per_path = 15
        
        # Training Parameters
        self.max_text_len = 512
        self.max_target_len = 64
        self.batch_size = 4
        self.grad_accum_steps = 4
        self.epochs = 100
        self.lr = 2e-5
        self.weight_decay = 0.01
        self.dropout = 0.1
        self.patience = 10
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.seed = 42

    @property
    def total_qubits(self): return self.n_streams * self.qubits_per_path
    
    @property
    def total_gates(self): return self.n_streams * self.gates_per_path

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
print(f"SOTA Pipeline Initialized | Device: {cfg.device} | Total Qubits: {cfg.total_qubits}")

from transformers import utils
utils.logging.set_verbosity_error()

C:\Users\tejes\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SOTA Pipeline Initialized | Device: cuda | Total Qubits: 32


In [2]:
class AdvancedVideoQADataset(Dataset):
    def __init__(self, df, tokenizer, cfg):
        self.samples = []
        print(f"Loading multimodal features from: {cfg.clip_path}")
        for _, row in tqdm(df.iterrows(), total=len(df)):
            v_id, st = row['video_id'], row['start_time']
            c_p = os.path.join(cfg.clip_path, str(v_id), f"{float(st):.1f}", 'embeddings.npy')
            e_p = os.path.join(cfg.eff_path, str(v_id), f"{float(st):.1f}", 'embeddings.npy')
            
            if os.path.exists(c_p) and os.path.exists(e_p):
                c, e = np.load(c_p)[:21], np.load(e_p)[:21]
                if len(c) < 21:
                    c = np.pad(c, ((0, 21-len(c)), (0, 0)))
                    e = np.pad(e, ((0, 21-len(e)), (0, 0)))
                
                target = tokenizer(str(row['Questions']), max_length=cfg.max_len, 
                                  padding='max_length', truncation=True, return_tensors='pt')
                labels = target['input_ids'].squeeze(0)
                labels[labels == tokenizer.pad_token_id] = -100
                
                self.samples.append({
                    'clip': torch.from_numpy(c).float(),
                    'eff': torch.from_numpy(e).float(),
                    'labels': labels
                })
        
        if not self.samples:
            print("WARNING: Zero valid samples found. Please check your data paths.")
        else:
            print(f"Dataset ready: {len(self.samples)} valid samples.")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

tokenizer = AutoTokenizer.from_pretrained(cfg.bart_model)
df = pd.read_excel(cfg.excel_path, sheet_name=cfg.sheet_name)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=cfg.seed)

train_loader = DataLoader(AdvancedVideoQADataset(train_df, tokenizer, cfg), batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(AdvancedVideoQADataset(val_df, tokenizer, cfg), batch_size=cfg.batch_size)

Loading multimodal features from: ..\embeddings/QCLIP


100%|██████████| 1602/1602 [00:01<00:00, 1506.43it/s]


Dataset ready: 1221 valid samples.
Loading multimodal features from: ..\embeddings/QCLIP


100%|██████████| 178/178 [00:00<00:00, 1477.24it/s]

Dataset ready: 135 valid samples.


In [3]:
def create_quantum_path(n_qubits, n_gates):
    dev = qml.device("default.qubit", wires=n_qubits)
    @qml.qnode(dev, interface="torch")
    def circuit(inputs):
        # RTheta configuration: High-expressibility RY-RZ Hardware-Efficient Ansatz
        # This circuit uses circular entanglement and alternating rotation layers
        # for optimal multimodal feature fusion.
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            
        param_idx = 0
        layer = 0
        while param_idx < n_gates:
            # Variational Layer: RY and RZ rotations
            for i in range(n_qubits):
                if param_idx < n_gates:
                    qml.RY(inputs[..., param_idx], wires=i)
                    param_idx += 1
                if param_idx < n_gates:
                    qml.RZ(inputs[..., param_idx], wires=i)
                    param_idx += 1
            
            # Entanglement Layer: Circular CNOT with alternating patterns
            if n_qubits > 1:
                for i in range(n_qubits):
                    if layer % 2 == 0:
                        qml.CNOT(wires=[i, (i + 1) % n_qubits])
                    else:
                        qml.CNOT(wires=[(i + 1) % n_qubits, i])
            layer += 1
            
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
    return circuit

class SOTAQuantumVideoBART(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.bart = BartForConditionalGeneration.from_pretrained(cfg.bart_model)
        self.clip_proj = nn.Linear(768, cfg.d_model)
        self.eff_proj = nn.Linear(1792, cfg.d_model)
        self.cross_attn = nn.MultiheadAttention(cfg.d_model, cfg.num_heads, batch_first=True)
        self.ln_fusion = nn.LayerNorm(cfg.d_model)
        self.q_down = nn.Linear(cfg.d_model, cfg.total_gates)
        self.q_paths = nn.ModuleList([
            qml.qnn.TorchLayer(create_quantum_path(cfg.qubits_per_path, cfg.gates_per_path), weight_shapes={})
            for _ in range(cfg.n_streams)
        ])
        self.q_up = nn.Linear(cfg.total_qubits, cfg.d_model)
        self.ln_q = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        
        def get_video_context(self, clip_feats, eff_feats):
        batch, seq, _ = clip_feats.shape
        c, e = self.clip_proj(clip_feats), self.eff_proj(eff_feats)
        fused, _ = self.cross_attn(query=e, key=c, value=c)
        fused = self.ln_fusion(fused + e) 
        q_params = self.q_down(fused).reshape(-1, self.cfg.total_gates)
        q_outputs = []
        for i in range(self.cfg.n_streams):
            start = i * self.cfg.gates_per_path
            end = (i + 1) * self.cfg.gates_per_path
            q_outputs.append(self.q_paths[i](q_params[:, start:end]))
        q_combined = torch.cat(q_outputs, dim=-1).reshape(batch, seq, self.cfg.total_qubits)
        return self.ln_q(fused + self.dropout(self.q_up(q_combined)))

    def forward(self, clip_feats, eff_feats, input_ids, attention_mask, labels=None):
        text_outputs = self.bart.model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_hidden = text_outputs.last_hidden_state
        video_ctx = self.get_video_context(clip_feats, eff_feats)
        
        if not hasattr(self, 'multimodal_cross_attn'):
            self.multimodal_cross_attn = nn.MultiheadAttention(self.cfg.d_model, self.cfg.num_heads, batch_first=True).to(self.cfg.device)
            self.ln_multimodal = nn.LayerNorm(self.cfg.d_model).to(self.cfg.device)

        mm_fused, _ = self.multimodal_cross_attn(query=text_hidden, key=video_ctx, value=video_ctx)
        final_hidden = self.ln_multimodal(text_hidden + self.dropout(mm_fused))
        
        return self.bart(encoder_outputs=BaseModelOutput(last_hidden_state=final_hidden), labels=labels, return_dict=True)

    def generate(self, clip_feats, eff_feats, input_ids, attention_mask, **kwargs):
        text_outputs = self.bart.model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_hidden = text_outputs.last_hidden_state
        video_ctx = self.get_video_context(clip_feats, eff_feats)
        
        if not hasattr(self, 'multimodal_cross_attn'):
            self.multimodal_cross_attn = nn.MultiheadAttention(self.cfg.d_model, self.cfg.num_heads, batch_first=True).to(self.cfg.device)
            self.ln_multimodal = nn.LayerNorm(self.cfg.d_model).to(self.cfg.device)

        mm_fused, _ = self.multimodal_cross_attn(query=text_hidden, key=video_ctx, value=video_ctx)
        final_hidden = self.ln_multimodal(text_hidden + self.dropout(mm_fused))
        
        return self.bart.generate(encoder_outputs=BaseModelOutput(last_hidden_state=final_hidden), **kwargs)

Loading weights: 100%|██████████| 259/259 [00:00<00:00, 15230.84it/s]


In [4]:
import evaluate
import numpy as np
from collections import Counter
from bert_score import score as bert_score_fn

# Load base metrics
rouge = evaluate.load('rouge',quiet=True)
bleu = evaluate.load('bleu',quiet=True)
meteor = evaluate.load('meteor',quiet=True)

def compute_distinct_n(predictions, n):
    tokens = [t for p in predictions for t in p.split()]
    if not tokens: return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    if not ngrams: return 0.0
    return len(set(ngrams)) / len(ngrams)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = GradScaler('cuda')
best_rouge_l = 0
epochs_no_improve = 0

wandb.init(project="Quantum-Video-SOTA-Final", name=f"Paths-{cfg.n_streams}-QperPath-{cfg.qubits_per_path}")

for epoch in range(cfg.epochs):
    model.train()
    total_train_loss, train_steps = 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(pbar):
        c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
        in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
        with autocast('cuda'):
            loss = model(c, e, in_ids, attn_mask, l).loss / max(1, cfg.grad_accum_steps)
        scaler.scale(loss).backward()
        if (i + 1) % cfg.grad_accum_steps == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        total_train_loss += loss.item() * cfg.grad_accum_steps
        train_steps += 1
        pbar.set_postfix({'loss': total_train_loss / max(1, train_steps)})
    
    model.eval()
    all_preds, all_labels, val_loss, val_steps = [], [], 0, 0
    with torch.no_grad():
        for batch in val_loader:
            c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
            in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
            outputs = model(c, e, in_ids, attn_mask, l)
            val_loss += outputs.loss.item(); val_steps += 1
            gen_ids = model.generate(c, e, in_ids, attn_mask, max_length=cfg.max_target_len, num_beams=5, early_stopping=True)
            all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
            all_labels.extend([tokenizer.decode(g[g != -100], skip_special_tokens=True) for g in l])

    avg_train = total_train_loss / max(1, train_steps)
    avg_val = val_loss / max(1, val_steps)
    metrics_log = {"epoch": epoch + 1, "train_loss": avg_train, "val_loss": avg_val}
    
    if all_preds:
        r_res = rouge.compute(predictions=all_preds, references=all_labels)
        b1_res = bleu.compute(predictions=all_preds, references=all_labels, max_order=1)
        b_res = bleu.compute(predictions=all_preds, references=all_labels)
        m_res = meteor.compute(predictions=all_preds, references=all_labels)
        try:
            P, R, F1 = bert_score_fn(all_preds, all_labels, model_type="distilbert-base-uncased", lang="en", device=cfg.device.type)
            bs_f1 = F1.mean().item()
        except: bs_f1 = 0.0
        
        dist1 = compute_distinct_n(all_preds, 1)
        dist2 = compute_distinct_n(all_preds, 2)
        metrics_log.update({"rougeL": r_res['rougeL'], "bleu1": b1_res['bleu'], "bleu": b_res['bleu'], "meteor": m_res['meteor'], "bert_score": bs_f1, "distinct1": dist1, "distinct2": dist2})
        print(f"\\nEpoch {epoch+1} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")
        print(f"R-L: {r_res['rougeL']:.4f} | B-1: {b1_res['bleu']:.4f} | Meteor: {m_res['meteor']:.4f} | BERT-S: {bs_f1:.4f}| Distinct1: {dist1:.4f} | Distinct2: {dist2:.4f}")
    
    wandb.log(metrics_log)
    if all_preds and metrics_log['rougeL'] > best_rouge_l:
        best_rouge_l = metrics_log['rougeL']; torch.save(model.state_dict(), 'best_model.pt'); epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= cfg.patience: break
wandb.finish()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tejes\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tejes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\tejes\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tejes\_netrc.
wandb: Currently logged in as: tejeshwarsingh1205 (tejeshwarsingh1205-indian-institute-of-technology-patna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1:   0%|          | 0/39 [00:00<?, ?it/s]c:\Users\tejes\OneDrive\Desktop\College Work\Sem 6\Capstone\capstone_env\Lib\site-packages\pennylane\ops\qubit\parametric_ops_single_qubit.py:347: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\EmptyTensor.cpp:56.)
  c = (1 + 0j) * c
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7143.01it/s]


BERTScore error: BertTokenizer has no attribute build_inputs_with_special_tokens
\nEpoch 1 | Train: 8.5184 | Val: 6.5246
R-L: 0.0000 | B-1: 0.0000 | Meteor: 0.0000 | BERT-S: 0.0000| Distinct1: 0.0000 | Distinct2: 0.0000


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10954.05it/s]


BERTScore error: BertTokenizer has no attribute build_inputs_with_special_tokens
\nEpoch 2 | Train: 6.3147 | Val: 6.5432
R-L: 0.0000 | B-1: 0.0000 | Meteor: 0.0000 | BERT-S: 0.0000| Distinct1: 0.0000 | Distinct2: 0.0000


...

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9220.07it/s]


\nEpoch 22 | Train: 2.1376 | Val: 5.6512
R-L: 0.2146 | B-1: 0.2301 | Meteor: 0.1647 | BERT-S: 0.7626| Distinct1: 0.2657 | Distinct2: 0.6030


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11103.97it/s]


\nEpoch 23 | Train: 2.0518 | Val: 5.8813
R-L: 0.1986 | B-1: 0.2170 | Meteor: 0.1624 | BERT-S: 0.7613| Distinct1: 0.2237 | Distinct2: 0.5310


bert_score,▁▁▁▁▇█▁████████████████
bleu,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▅▁▁▁▁▁█▅
bleu1,▁▁▁▂▃▆▄▅▇▇██▇█▇▇███▆██▇
distinct1,▁▁█▃▄▄▅▄▅▅▅▅▆▅▆▅▆▇▆▆▆▆▅
distinct2,▁▁█▃▅▅▅▅▅▅▅▅▆▅▆▅▆▆▆▆▆▆▅
epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
meteor,▁▁▂▂▃▅▄▄▆▆▇▇▇▇▇▇███▇██▇
rougeL,▁▁▂▂▅▆▅▅▇▇▇███▇▇█▇████▇
train_loss,█▆▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val_loss,██▅▅▂▂▂▂▁▁▁▁▂▂▂▃▃▂▃▃▃▃▄
bert_score,0.76132
